# Análise de Desempenho dos Algoritmos de Ordenação

Este notebook gera gráficos dinâmicos a partir do CSV produzido pelo projeto Java.

Algoritmos analisados:

- Bubble Sort
- Selection Sort
- Merge Sort
- Quick Sort

Cada algoritmo possui versão serial e versão paralela.

In [1]:
# Caso necessário, instale as bibliotecas:
# !pip install pandas plotly ipywidgets kaleido

import re
from pathlib import Path

import pandas as pd
import plotly.express as px


## 1. Carregar CSV

Por padrão, o Java salva o resultado em `out/resultados_ordenacao.csv`.

In [2]:
csv_path = Path("out/resultados_ordenacao.csv")
images_path = Path("images")
images_path.mkdir(exist_ok=True)


def slugify(value):
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def configure_figure(fig, legend_title):
    fig.update_layout(
        template="plotly_white",
        xaxis_title="Tamanho do vetor",
        yaxis_title=fig.layout.yaxis.title.text,
        legend_title=legend_title,
        width=1400,
        height=720,
        margin=dict(t=110, r=320, b=90, l=90),
        title=dict(x=0.5, xanchor="center"),
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
            tracegroupgap=6
        )
    )
    fig.update_annotations(font=dict(size=11))
    fig.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text.split("=")[-1].replace("_", " ")
        )
    )
    fig.update_xaxes(tickangle=0, automargin=True)
    fig.update_yaxes(automargin=True)
    return fig


PNG_EXPORT_AVAILABLE = True


def save_figure(fig, name):
    slug = slugify(name)
    png_file = images_path / f"{slug}.png"

    global PNG_EXPORT_AVAILABLE
    if PNG_EXPORT_AVAILABLE:
        try:
            fig.write_image(png_file, scale=2)
            print(f"- {png_file}")
        except ValueError:
            PNG_EXPORT_AVAILABLE = False
            print(
                "PNG nao exportado porque o pacote kaleido nao esta instalado. "
                "Execute: pip install kaleido"
            )


if not csv_path.exists():
    raise FileNotFoundError(
        "Arquivo out/resultados_ordenacao.csv nao encontrado. "
        "Execute primeiro o projeto Java para gerar o CSV."
    )

df = pd.read_csv(csv_path)
df.head()


,algorithm,base_algorithm,execution_type,threads,size,data_type,sample,time_ns,time_ms,sorted
0,BubbleSortSerial,BubbleSort,SERIAL,1,1000,RANDOM,1,5958696,5.958696,True
1,BubbleSortParallel,BubbleSort,PARALLEL,2,1000,RANDOM,1,15432642,15.432642,True
2,BubbleSortParallel,BubbleSort,PARALLEL,4,1000,RANDOM,1,19419808,19.419808,True
3,BubbleSortParallel,BubbleSort,PARALLEL,8,1000,RANDOM,1,40137217,40.137217,True
4,BubbleSortParallel,BubbleSort,PARALLEL,16,1000,RANDOM,1,78021826,78.021826,True


## 2. Conferir estrutura dos dados

In [3]:
print("Total de linhas:", len(df))
print("Colunas:", list(df.columns))
print("\nAlgoritmos:")
print(df["algorithm"].unique())
print("\nAlgoritmos base:")
print(df["base_algorithm"].unique())
print("\nTipos de dados:")
print(df["data_type"].unique())
print("\nThreads:")
print(sorted(df["threads"].unique()))

Total de linhas: 1600
Colunas: ['algorithm', 'base_algorithm', 'execution_type', 'threads', 'size', 'data_type', 'sample', 'time_ns', 'time_ms', 'sorted']

Algoritmos:
<StringArray>
[     'BubbleSortSerial',    'BubbleSortParallel',   'SelectionSortSerial',
 'SelectionSortParallel',       'MergeSortSerial',     'MergeSortParallel',
       'QuickSortSerial',     'QuickSortParallel']
Length: 8, dtype: str

Algoritmos base:
<StringArray>
['BubbleSort', 'SelectionSort', 'MergeSort', 'QuickSort']
Length: 4, dtype: str

Tipos de dados:
<StringArray>
['RANDOM', 'SORTED', 'REVERSED', 'NEARLY_SORTED']
Length: 4, dtype: str

Threads:
[np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16)]


## 3. Estatísticas: média, desvio padrão e amostras

In [4]:
summary = (
    df.groupby(["algorithm", "base_algorithm", "execution_type", "threads", "size", "data_type"], as_index=False)
      .agg(
          mean_time_ms=("time_ms", "mean"),
          std_time_ms=("time_ms", "std"),
          samples=("time_ms", "count")
      )
)

summary.head()

,algorithm,base_algorithm,execution_type,threads,size,data_type,mean_time_ms,std_time_ms,samples
0,BubbleSortParallel,BubbleSort,PARALLEL,2,1000,NEARLY_SORTED,4.783781,0.047828,5
1,BubbleSortParallel,BubbleSort,PARALLEL,2,1000,RANDOM,7.294764,4.560881,5
2,BubbleSortParallel,BubbleSort,PARALLEL,2,1000,REVERSED,4.848958,0.642635,5
3,BubbleSortParallel,BubbleSort,PARALLEL,2,1000,SORTED,5.023052,1.033126,5
4,BubbleSortParallel,BubbleSort,PARALLEL,2,2000,NEARLY_SORTED,10.254483,0.918156,5


## 4. Tempo médio geral por algoritmo

In [5]:
fig = px.line(
    summary,
    x="size",
    y="mean_time_ms",
    color="algorithm",
    line_dash="data_type",
    facet_col="execution_type",
    markers=True,
    hover_data=["threads", "std_time_ms", "samples"],
    title="Tempo medio de execucao por tamanho da entrada"
)

fig.update_layout(yaxis_title="Tempo medio (ms)")
configure_figure(fig, "Algoritmo")
save_figure(fig, "tempo_medio_por_tamanho_execucao")
fig.show()


- images/tempo_medio_por_tamanho_execucao.png
- images/tempo_medio_por_tamanho_execucao.html


## 5. Comparação por tipo de entrada

In [6]:
tipo_entrada = "RANDOM"

filtered = summary[summary["data_type"] == tipo_entrada]

fig = px.line(
    filtered,
    x="size",
    y="mean_time_ms",
    color="algorithm",
    line_dash="threads",
    markers=True,
    hover_data=["execution_type", "std_time_ms", "samples"],
    title=f"Tempo medio por tamanho da entrada - {tipo_entrada}"
)

fig.update_layout(yaxis_title="Tempo medio (ms)")
configure_figure(fig, "Algoritmo / Threads")
save_figure(fig, f"tempo_medio_{tipo_entrada}")
fig.show()


- images/tempo_medio_random.png
- images/tempo_medio_random.html


## 6. Comparação apenas dos algoritmos paralelos

In [7]:
parallel = summary[summary["execution_type"] == "PARALLEL"]

fig = px.line(
    parallel,
    x="size",
    y="mean_time_ms",
    color="base_algorithm",
    line_dash="threads",
    facet_col="data_type",
    markers=True,
    hover_data=["algorithm", "std_time_ms", "samples"],
    title="Versoes paralelas: comparacao por quantidade de threads"
)

fig.update_layout(yaxis_title="Tempo medio (ms)")
configure_figure(fig, "Algoritmo / Threads")
save_figure(fig, "versoes_paralelas_por_threads")
fig.show()


- images/versoes_paralelas_por_threads.png
- images/versoes_paralelas_por_threads.html


## 7. Comparação por algoritmo específico

In [8]:
algoritmo = "MergeSort"  # Opcoes: BubbleSort, SelectionSort, MergeSort, QuickSort

alg_df = summary[summary["base_algorithm"] == algoritmo]

fig = px.line(
    alg_df,
    x="size",
    y="mean_time_ms",
    color="execution_type",
    line_dash="threads",
    facet_col="data_type",
    markers=True,
    hover_data=["algorithm", "std_time_ms", "samples"],
    title=f"Comparacao Serial x Paralelo - {algoritmo}"
)

fig.update_layout(yaxis_title="Tempo medio (ms)")
configure_figure(fig, "Execucao / Threads")
save_figure(fig, f"comparacao_serial_paralelo_{algoritmo}")
fig.show()


- images/comparacao_serial_paralelo_mergesort.png
- images/comparacao_serial_paralelo_mergesort.html


## 8. Cálculo de Speedup

Speedup:

\[
Speedup = \frac{Tempo\ Serial}{Tempo\ Paralelo}
\]

Valores maiores que 1 indicam ganho de desempenho.

In [9]:
serial = summary[summary["execution_type"] == "SERIAL"].copy()
parallel = summary[summary["execution_type"] == "PARALLEL"].copy()

speedup = parallel.merge(
    serial[["base_algorithm", "size", "data_type", "mean_time_ms"]],
    on=["base_algorithm", "size", "data_type"],
    suffixes=("_parallel", "_serial")
)

speedup["speedup"] = speedup["mean_time_ms_serial"] / speedup["mean_time_ms_parallel"]
speedup["efficiency"] = speedup["speedup"] / speedup["threads"]

speedup.head()

,algorithm,base_algorithm,execution_type,threads,size,data_type,mean_time_ms_parallel,std_time_ms,samples,mean_time_ms_serial,speedup,efficiency
0,BubbleSortParallel,BubbleSort,PARALLEL,2,1000,NEARLY_SORTED,4.783781,0.047828,5,0.181841,0.038012,0.019006
1,BubbleSortParallel,BubbleSort,PARALLEL,2,1000,RANDOM,7.294764,4.560881,5,1.851261,0.253779,0.126890
2,BubbleSortParallel,BubbleSort,PARALLEL,2,1000,REVERSED,4.848958,0.642635,5,1.084061,0.223566,0.111783
3,BubbleSortParallel,BubbleSort,PARALLEL,2,1000,SORTED,5.023052,1.033126,5,0.002616,0.000521,0.000260
4,BubbleSortParallel,BubbleSort,PARALLEL,2,2000,NEARLY_SORTED,10.254483,0.918156,5,0.808400,0.078834,0.039417


## 9. Gráfico de Speedup

In [10]:
fig = px.line(
    speedup,
    x="size",
    y="speedup",
    color="base_algorithm",
    line_dash="threads",
    facet_col="data_type",
    markers=True,
    hover_data=["mean_time_ms_serial", "mean_time_ms_parallel", "efficiency"],
    title="Speedup das versoes paralelas em relacao as versoes seriais"
)

fig.add_hline(y=1, line_dash="dot")
fig.update_layout(yaxis_title="Speedup")
configure_figure(fig, "Algoritmo / Threads")
save_figure(fig, "speedup_versoes_paralelas")
fig.show()


- images/speedup_versoes_paralelas.png
- images/speedup_versoes_paralelas.html


## 10. Gráfico de Eficiência Paralela

In [11]:
fig = px.line(
    speedup,
    x="size",
    y="efficiency",
    color="base_algorithm",
    line_dash="threads",
    facet_col="data_type",
    markers=True,
    hover_data=["speedup"],
    title="Eficiencia paralela"
)

fig.update_layout(yaxis_title="Eficiencia")
configure_figure(fig, "Algoritmo / Threads")
save_figure(fig, "eficiencia_paralela")
fig.show()


- images/eficiencia_paralela.png
- images/eficiencia_paralela.html


## 11. Exportar tabelas resumidas

In [12]:
summary.to_csv("out/resumo_estatistico.csv", index=False)
speedup.to_csv("out/speedup_eficiencia.csv", index=False)

print("Arquivos gerados:")
print("- out/resumo_estatistico.csv")
print("- out/speedup_eficiencia.csv")
print("- images/")


Arquivos gerados:
- out/resumo_estatistico.csv
- out/speedup_eficiencia.csv
- images/
